# DeepSeek-OCR to RAG Formatter (v2)

This version includes **Row Merging**: If a row doesn't have an ID (الرقم الترتيبي), its content is automatically merged into the previous row. This ensures logical entries like item 07 stay as a single block.


In [4]:
print("hello")

hello


In [3]:
import re
from html.parser import HTMLParser


class TableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tables = []
        self.current_table = []
        self.current_row = []
        self.current_cell = ""
        self.in_td = False

    def handle_starttag(self, tag, attrs):
        if tag == "table":
            self.current_table = []
        elif tag == "tr":
            self.current_row = []
        elif tag == "td":
            self.in_td = True
            self.current_cell = ""

    def handle_endtag(self, tag):
        if tag == "td":
            self.in_td = False
            self.current_row.append(self.current_cell.strip())
        elif tag == "tr":
            if any(self.current_row):
                self.current_table.append(self.current_row)
        elif tag == "table":
            self.tables.append(self.current_table)

    def handle_data(self, data):
        if self.in_td:
            self.current_cell += data


def clean_deepseek_output(text):
    """Removes grounding tags and coordinate markers."""
    text = re.sub(r"<\|ref\|>.*?<\|/ref\|>", "", text)
    text = re.sub(r"<\|det\|>.*?<\|/det\|>", "", text)
    text = re.sub(r"<\|.*?\|>", "", text)
    return text.strip()


def html_to_rag(html_input):
    """Converts HTML tables to RAG blocks with smart row merging."""
    parser = TableParser()
    parser.feed(html_input)

    final_text = html_input
    table_matches = re.findall(r"<table>.*?</table>", html_input, re.DOTALL)

    for i, table_html in enumerate(table_matches):
        if i >= len(parser.tables):
            break

        grid = parser.tables[i]
        if not grid or len(grid) < 2:
            continue

        headers = grid[0]
        # Merging logic: group rows by ID
        merged_rows = []
        for row in grid[1:]:
            if all(not c or c == "..." for c in row):
                continue

            # If ID (col 0) is empty, merge with previous entry
            if not row[0].strip() and merged_rows:
                prev = merged_rows[-1]
                for j in range(1, min(len(row), len(prev))):
                    if row[j].strip():
                        prev[j] = (prev[j] + " " + row[j].strip()).strip()
            else:
                merged_rows.append([c.strip() for c in row])

        blocks = []
        for row in merged_rows:
            block = ["نوع المحتوى: جدول"]
            for j, cell in enumerate(row):
                if not cell or cell == "...":
                    continue
                header = headers[j] if j < len(headers) else f"Column {j+1}"
                block.append(f"{header}: {cell}")
            if len(block) > 1:
                blocks.append("\n".join(block))

        rag_replacement = "\n\n".join(blocks)
        final_text = final_text.replace(table_html, rag_replacement)

    return final_text


print("Advanced Formatter Ready.")

Advanced Formatter Ready.


### 2. Extract and Auto-Format

Prompt changed to "Extract text and tables" to align with pdf2text goals.


In [4]:
import ollama
import time

model_name = "deepseek-ocr:latest"
img_path = "ollama_test_table.png"

print(f"Processing {img_path}...\n")
start_time = time.time()

stream = ollama.generate(
    model=model_name,
    prompt="<|grounding|>Extract the document text and tables.",
    images=[img_path],
    stream=True,
    options={"temperature": 0, "num_ctx": 8192, "repeat_penalty": 1.5},
)

raw_response = ""
for chunk in stream:
    print(chunk["response"], end="", flush=True)
    raw_response += chunk["response"]

print(f"\n\n--- OCR Finished ({time.time() - start_time:.2f}s) ---")

no_tags = clean_deepseek_output(raw_response)
rag_ready_text = html_to_rag(no_tags)

print("\n--- FINAL RAG-READY OUTPUT ---")
print(rag_ready_text)

Processing ollama_test_table.png...

<|ref|>table_caption<|/ref|><|det|>[[437, 5, 561, 31]]<|/det|>
«الجنح 

<|ref|>table<|/ref|><|det|>[[50, 37, 969, 632]]<|/det|>
<table><tr><td>الرقم التربيعي</td><td>الجنحة</td><td>النقط الواجب خصمها</td></tr><tr><td>01</td><td>...</td><td>...</td></tr><tr><td>...</td><td>...</td><td>...</td></tr><tr><td>07</td><td>سياقة مركبة تحت تأثير الكحول أو تحت تأثير المواد المخدرة</td><td>6</td></tr><tr><td></td><td>- رفض الخضوع للرائز المشار إليه في المادة 207 أدناه أو للتحققات أو لاختبارات الكشف المنصوص عليها في المادتين 208 و213 أدناه</td><td></td></tr><tr><td>...</td><td>...</td><td>...</td></tr><tr><td>13</td><td>السائق الذي وجه إليه الأمر بالوقوف وامتنع من تنفيذه أولم يحترم الأمر بتوقيف المركبة أو رفض سياقة مركبته أو العمل على سياقها إلى المحجز أو رفض الامتثال للأوامر القانونية الصادرة إليه</td><td>2</td></tr><tr><td>...</td><td>...</td><td>...</td></tr><tr><td>18</td><td>...</td><td>...</td></tr></table>

<|ref|>table_caption<|/ref|><|det|>[[405, 660, 